# Predictive Maintenance: Interactive Streamlit Dashboard

This notebook turns the analysis from `predictive_maintenance.ipynb` into an interactive dashboard. It writes two files next to the notebook and then launches the app:

| File | Purpose |
|---|---|
| `pm_core.py` | Data cleaning, feature engineering, models, FDR and cost-based threshold selection, Isolation Forest (same logic and same random seeds as the main notebook) |
| `app.py` | The Streamlit user interface |

**Expected folder layout** (same as the main notebook):
```
project/
├── data/ai4i2020.csv
└── notebooks/
    ├── streamlit_dashboard.ipynb   <- this notebook
    ├── pm_core.py                  <- created by cell 2
    └── app.py                      <- created by cell 3
```
If the CSV is somewhere else, upload it from the dashboard sidebar.

## Which requirement is covered where

| Requirement | Dashboard tab |
|---|---|
| Classify failure types (Tool Wear, Heat Dissipation, Power Failure, Overstrain) | Models & threshold, Live prediction |
| Focus on the **False Discovery Rate**, minimise false positives even if accuracy drops | Models & threshold (FDR rule, default vs tuned accuracy side by side) |
| Cost-sensitive learning | Cost-sensitive (editable costs, FDR rule vs cost rule, sensitivity to cost ratio) |
| Anomaly detection | Anomaly detection (Isolation Forest, threshold table, comparison with supervised models) |
| Feature engineering | Overview (engineered features in the box plots), Lead indicator |
| **Bonus:** lead sensor of failure | Lead indicator |
| User enters data and the model predicts | Live prediction (form, presets, batch CSV) |


In [7]:
# Only needed once. Skip if these packages are already installed.
%pip install -q "streamlit>=1.50" plotly xgboost scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
%%writefile pm_core.py
"""Core logic for the Predictive Maintenance dashboard.

Mirrors predictive_maintenance.ipynb: same cleaning, features, split, models,
FDR-based threshold selection, cost-sensitive selection and Isolation Forest.
"""
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (accuracy_score, average_precision_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# ----------------------------------------------------------------------------
# Constants
# ----------------------------------------------------------------------------
SENSORS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
ENGINEERED = ["Temp difference [K]", "Power [W]", "Wear x Torque"]
CATEGORICAL = ["Type"]
NUMERICAL = SENSORS + ENGINEERED
FEATURES = CATEGORICAL + NUMERICAL

ALL_MODES = ["TWF", "HDF", "PWF", "OSF", "RNF"]
MAIN_MODES = ["TWF", "HDF", "PWF", "OSF"]
FAILURE_NAMES = {
    "TWF": "Tool Wear",
    "HDF": "Heat Dissipation",
    "PWF": "Power Failure",
    "OSF": "Overstrain",
}
THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
PERCENTILES = [50, 60, 70, 80, 85, 90, 95, 97, 98, 99]
DATA_CANDIDATES = ["../data/ai4i2020.csv", "data/ai4i2020.csv", "ai4i2020.csv"]


# ----------------------------------------------------------------------------
# Data
# ----------------------------------------------------------------------------
def find_dataset():
    """Look for ai4i2020.csv in the usual places. Returns a Path or None."""
    here = Path(__file__).resolve().parent
    for candidate in DATA_CANDIDATES:
        for base in (Path.cwd(), here):
            path = (base / candidate).resolve()
            if path.exists():
                return path
    return None


def add_features(df):
    """Physics-based features (same formulas as the notebook)."""
    df = df.copy()
    df["Temp difference [K]"] = (
        df["Process temperature [K]"] - df["Air temperature [K]"]
    )
    df["Power [W]"] = (
        df["Torque [Nm]"] * df["Rotational speed [rpm]"] * 2 * np.pi / 60
    )
    df["Wear x Torque"] = df["Tool wear [min]"] * df["Torque [Nm]"]
    return df


def clean_and_label(raw):
    """Remove inconsistent + RNF rows and build the 5-class 'Failure Type'."""
    df = raw.copy()
    df["failure_mode_count"] = df[ALL_MODES].sum(axis=1)

    inconsistent = (df["Machine failure"] == 1) & (df["failure_mode_count"] == 0)
    n_inconsistent = int(inconsistent.sum())
    df = df[~inconsistent].copy()

    n_rnf = int((df["RNF"] == 1).sum())
    df = df[df["RNF"] == 0].copy()

    df["Failure Type"] = "No Failure"
    has_failure = df[MAIN_MODES].sum(axis=1) > 0
    df.loc[has_failure, "Failure Type"] = (
        df.loc[has_failure, MAIN_MODES].idxmax(axis=1).map(FAILURE_NAMES)
    )
    df = add_features(df)

    info = {
        "raw_rows": len(raw),
        "clean_rows": len(df),
        "inconsistent_removed": n_inconsistent,
        "rnf_removed": n_rnf,
        "multi_mode_rows": int((df["failure_mode_count"] > 1).sum()),
    }
    return df, info


def correlation_by_type(df):
    """Correlation of each feature with 'this failure type vs normal machines'."""
    columns = SENSORS + ENGINEERED
    out = pd.DataFrame(index=columns, columns=list(FAILURE_NAMES.values()), dtype=float)
    for failure_type in FAILURE_NAMES.values():
        subset = df[df["Failure Type"].isin(["No Failure", failure_type])]
        target = (subset["Failure Type"] == failure_type).astype(int)
        out[failure_type] = subset[columns].corrwith(target)
    return out


def lead_sensors(corr):
    """Strongest raw sensor per failure type + average ranking (raw sensors only)."""
    raw = corr.loc[SENSORS]
    rows = []
    for failure_type in raw.columns:
        best = raw[failure_type].abs().idxmax()
        rows.append({
            "Failure Type": failure_type,
            "Lead sensor": best,
            "Correlation": raw.loc[best, failure_type],
        })
    ranking = raw.abs().mean(axis=1).sort_values(ascending=False)
    return pd.DataFrame(rows), ranking


# ----------------------------------------------------------------------------
# Metrics (same definitions as the notebook)
# ----------------------------------------------------------------------------
def predict_with_threshold(proba, threshold, no_failure_index):
    """Alarm only if the best failure class has probability >= threshold."""
    pred = np.full(len(proba), no_failure_index)
    failure_proba = proba.copy()
    failure_proba[:, no_failure_index] = 0
    best_class = failure_proba.argmax(axis=1)
    best_score = failure_proba.max(axis=1)
    alarm = best_score >= threshold
    pred[alarm] = best_class[alarm]
    return pred


def failure_metrics(y_true, y_pred, no_failure_index):
    alarms = y_pred != no_failure_index
    actual = y_true != no_failure_index
    correct = alarms & (y_pred == y_true)

    fp = int((alarms & ~actual).sum())
    wrong_type = int((alarms & actual & (y_pred != y_true)).sum())
    fn = int((actual & ~alarms).sum())

    n_alarms = int(alarms.sum())
    n_correct = int(correct.sum())
    n_actual = int(actual.sum())

    precision = n_correct / n_alarms if n_alarms else 0.0
    recall = n_correct / n_actual if n_actual else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    fdr = (n_alarms - n_correct) / n_alarms if n_alarms else 0.0

    return {
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "False Positives": fp,
        "Wrong Type": wrong_type,
        "False Negatives": fn,
        "FDR": fdr,
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Alarms": n_alarms,
    }


def total_cost(m, c_false_alarm, c_wrong_type, c_missed):
    return (m["False Positives"] * c_false_alarm
            + m["Wrong Type"] * c_wrong_type
            + m["False Negatives"] * c_missed)


def threshold_table(y_true, proba, no_failure_index, thresholds=THRESHOLDS):
    rows = []
    for t in thresholds:
        pred = predict_with_threshold(proba, t, no_failure_index)
        m = failure_metrics(y_true, pred, no_failure_index)
        m["Threshold"] = t
        rows.append(m)
    cols = ["Threshold", "Precision", "Recall", "F1-score", "False Positives",
            "Wrong Type", "False Negatives", "FDR", "Accuracy"]
    return pd.DataFrame(rows)[cols]


def add_cost_column(table, c_false_alarm, c_wrong_type, c_missed):
    table = table.copy()
    table["Total Cost"] = (table["False Positives"] * c_false_alarm
                           + table["Wrong Type"] * c_wrong_type
                           + table["False Negatives"] * c_missed)
    return table


def select_fdr_rule(table, min_recall):
    """Lowest FDR with Recall >= min_recall (ties: fewer FP, higher recall)."""
    eligible = table[table["Recall"] >= min_recall]
    if eligible.empty:
        return None
    return eligible.sort_values(
        by=["FDR", "False Positives", "Recall"], ascending=[True, True, False]
    ).iloc[0]


def select_cost_rule(table_with_cost):
    """Lowest total cost (ties: fewer false positives)."""
    return table_with_cost.sort_values(
        by=["Total Cost", "False Positives"], ascending=[True, True]
    ).iloc[0]


# ----------------------------------------------------------------------------
# Training
# ----------------------------------------------------------------------------
def train_bundle(raw, seed=42):
    """Train everything the dashboard needs. Returns a dict (cached by the app)."""
    df, info = clean_and_label(raw)

    X = df[FEATURES].copy()
    encoder = LabelEncoder()
    y = encoder.fit_transform(df["Failure Type"])
    class_names = list(encoder.classes_)
    nf = class_names.index("No Failure")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=seed, stratify=y_train)

    # Random Forest
    rf_pre = ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("numerical", StandardScaler(), NUMERICAL),
    ])
    rf = Pipeline([
        ("preprocessor", rf_pre),
        ("classifier", RandomForestClassifier(
            n_estimators=200, class_weight="balanced", random_state=seed, n_jobs=-1)),
    ])
    rf.fit(X_tr, y_tr)

    # XGBoost (class-balanced sample weights)
    xgb_pre = ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("numerical", "passthrough", NUMERICAL),
    ])
    xgb = Pipeline([
        ("preprocessor", xgb_pre),
        ("classifier", XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            objective="multi:softprob", eval_metric="mlogloss",
            random_state=seed, n_jobs=-1)),
    ])
    weights = compute_sample_weight(class_weight="balanced", y=y_tr)
    xgb.fit(X_tr, y_tr, classifier__sample_weight=weights)

    models = {"Random Forest": rf, "XGBoost": xgb}
    proba = {
        name: {"val": m.predict_proba(X_val), "test": m.predict_proba(X_test)}
        for name, m in models.items()
    }

    # Isolation Forest: trained on normal machines only
    iso_pre = ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("numerical", StandardScaler(), NUMERICAL),
    ])
    iso = Pipeline([
        ("preprocessor", iso_pre),
        ("detector", IsolationForest(n_estimators=200, random_state=seed, n_jobs=-1)),
    ])
    iso.fit(X_tr[y_tr == nf])
    anomaly = {
        "val": -iso.score_samples(X_val),
        "test": -iso.score_samples(X_test),
    }

    corr = correlation_by_type(df)

    return {
        "df": df, "info": info, "class_names": class_names, "nf": nf,
        "X_val": X_val, "X_test": X_test, "y_val": y_val, "y_test": y_test,
        "models": models, "proba": proba, "iso": iso, "anomaly": anomaly,
        "corr": corr,
    }


# ----------------------------------------------------------------------------
# Anomaly detection helpers
# ----------------------------------------------------------------------------
def anomaly_table(bundle, min_recall):
    nf = bundle["nf"]
    yv = (bundle["y_val"] != nf).astype(int)
    score = bundle["anomaly"]["val"]
    rows = []
    for p in PERCENTILES:
        thr = np.percentile(score[yv == 0], p)
        pred = (score >= thr).astype(int)
        tp = int(((pred == 1) & (yv == 1)).sum())
        fp = int(((pred == 1) & (yv == 0)).sum())
        fn = int(((pred == 0) & (yv == 1)).sum())
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            "Percentile": p, "Threshold": thr, "Precision": precision,
            "Recall": recall, "F1-score": f1, "False Positives": fp,
            "False Negatives": fn, "FDR": fp / (fp + tp) if (fp + tp) else 0.0,
        })
    table = pd.DataFrame(rows)
    eligible = table[table["Recall"] >= min_recall]
    best = None
    if not eligible.empty:
        best = eligible.sort_values(
            by=["FDR", "False Positives", "Recall"], ascending=[True, True, False]
        ).iloc[0]
    return table, best


def anomaly_test_result(bundle, threshold):
    nf = bundle["nf"]
    yt = (bundle["y_test"] != nf).astype(int)
    pred = (bundle["anomaly"]["test"] >= threshold).astype(int)
    tp = int(((pred == 1) & (yt == 1)).sum())
    fp = int(((pred == 1) & (yt == 0)).sum())
    fn = int(((pred == 0) & (yt == 1)).sum())
    return {
        "Precision": tp / (tp + fp) if (tp + fp) else 0.0,
        "Recall": tp / (tp + fn) if (tp + fn) else 0.0,
        "False Positives": fp, "False Negatives": fn,
        "FDR": fp / (fp + tp) if (fp + tp) else 0.0,
    }


def ranking_comparison(bundle):
    """Threshold-free comparison: ROC-AUC and Average Precision on the test set."""
    nf = bundle["nf"]
    yt = (bundle["y_test"] != nf).astype(int)
    scores = {"Isolation Forest": bundle["anomaly"]["test"]}
    for name in bundle["models"]:
        scores[name] = 1 - bundle["proba"][name]["test"][:, nf]
    rows = [{"Model": n, "ROC-AUC": roc_auc_score(yt, s),
             "Average Precision": average_precision_score(yt, s)}
            for n, s in scores.items()]
    return pd.DataFrame(rows)


def anomaly_auc_by_type(bundle):
    nf, y = bundle["nf"], bundle["y_test"]
    rows = []
    for idx, name in enumerate(bundle["class_names"]):
        if idx == nf:
            continue
        mask = (y == idx) | (y == nf)
        if (y == idx).sum() == 0:
            continue
        auc = roc_auc_score((y[mask] == idx).astype(int), bundle["anomaly"]["test"][mask])
        rows.append({"Failure Type": name, "ROC-AUC": auc})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# Single / batch prediction
# ----------------------------------------------------------------------------
def build_input_frame(rows):
    """rows: list of dicts with Type + the 5 raw sensors -> model-ready frame."""
    return add_features(pd.DataFrame(rows))[FEATURES]


def predict_frame(bundle, model_name, X, threshold):
    nf = bundle["nf"]
    proba = bundle["models"][model_name].predict_proba(X)
    pred = predict_with_threshold(proba, threshold, nf)
    labels = [bundle["class_names"][i] for i in pred]
    anomaly_score = -bundle["iso"].score_samples(X)
    return proba, pred, labels, anomaly_score


def physical_checks(row):
    """Documented AI4I 2020 failure rules, evaluated for one machine (sanity check)."""
    wear, torque = row["Tool wear [min]"], row["Torque [Nm]"]
    rpm = row["Rotational speed [rpm]"]
    diff = row["Process temperature [K]"] - row["Air temperature [K]"]
    power = torque * rpm * 2 * np.pi / 60
    strain_limit = {"L": 11000, "M": 12000, "H": 13000}[row["Type"]]
    return pd.DataFrame([
        {"Failure mode": "Tool Wear",
         "Rule": "Tool wear between 200 and 240 min",
         "Current value": f"{wear:.0f} min", "Triggered": 200 <= wear <= 240},
        {"Failure mode": "Heat Dissipation",
         "Rule": "Temp difference < 8.6 K and speed < 1380 rpm",
         "Current value": f"{diff:.1f} K, {rpm:.0f} rpm",
         "Triggered": bool(diff < 8.6 and rpm < 1380)},
        {"Failure mode": "Power Failure",
         "Rule": "Power < 3,500 W or > 9,000 W",
         "Current value": f"{power:,.0f} W", "Triggered": bool(power < 3500 or power > 9000)},
        {"Failure mode": "Overstrain",
         "Rule": f"Wear x Torque > {strain_limit:,} (type {row['Type']})",
         "Current value": f"{wear * torque:,.0f}", "Triggered": bool(wear * torque > strain_limit)},
    ])


Overwriting pm_core.py


In [9]:
%%writefile app.py
"""Industrial Predictive Maintenance dashboard (Streamlit).

Run with:  streamlit run app.py
"""
import io

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from sklearn.metrics import classification_report, confusion_matrix

import pm_core as pm

st.set_page_config(page_title="Predictive Maintenance", page_icon="🏭", layout="wide")


# ----------------------------------------------------------------------------
# Data + models (cached: trained once per session)
# ----------------------------------------------------------------------------
@st.cache_resource(show_spinner="Training models (about 10 seconds)...")
def get_bundle(csv_bytes, csv_path):
    raw = pd.read_csv(io.BytesIO(csv_bytes)) if csv_bytes else pd.read_csv(csv_path)
    return pm.train_bundle(raw)


with st.sidebar:
    st.title("🏭 Settings")
    uploaded = st.file_uploader("Dataset (optional)", type="csv",
                                help="Leave empty to use ai4i2020.csv from the data folder.")
    path = pm.find_dataset()
    if uploaded is None and path is None:
        st.error("ai4i2020.csv not found. Put it in ../data/ or upload it above.")
        st.stop()

    bundle = get_bundle(uploaded.getvalue() if uploaded else None, str(path))
    classes, nf = bundle["class_names"], bundle["nf"]

    st.subheader("Model")
    model_name = st.radio("Classifier", list(bundle["models"]), horizontal=True)
    min_recall = st.slider("Minimum recall (FDR rule)", 0.50, 0.95, 0.70, 0.05)

    st.subheader("Costs (relative units, assumed)")
    c_fa = st.number_input("False alarm", 0.0, 1000.0, 1.0, 1.0)
    c_wt = st.number_input("Wrong failure type", 0.0, 1000.0, 2.0, 1.0)
    c_mf = st.number_input("Missed failure", 0.0, 1000.0, 10.0, 1.0)

    # Validation tables for the chosen model (thresholds are always chosen on validation)
    val_table = pm.threshold_table(bundle["y_val"], bundle["proba"][model_name]["val"], nf)
    val_cost = pm.add_cost_column(val_table, c_fa, c_wt, c_mf)
    fdr_row = pm.select_fdr_rule(val_table, min_recall)
    cost_row = pm.select_cost_rule(val_cost)

    st.subheader("Operating threshold")
    options = ["FDR rule", "Cost rule", "Manual"]
    mode = st.radio("Choose by", options, help=(
        "FDR rule: lowest false discovery rate with recall above the minimum (validation set).\n\n"
        "Cost rule: lowest total cost (validation set)."))
    if mode == "FDR rule":
        if fdr_row is None:
            st.warning("No threshold reaches that recall, using the one with the best recall.")
            threshold = float(val_table.sort_values("Recall", ascending=False).iloc[0]["Threshold"])
        else:
            threshold = float(fdr_row["Threshold"])
    elif mode == "Cost rule":
        threshold = float(cost_row["Threshold"])
    else:
        threshold = st.slider("Threshold", 0.05, 0.95, 0.50, 0.05)
    st.metric("Active threshold", f"{threshold:.2f}")

st.title("Industrial Predictive Maintenance Dashboard")
st.caption("AI4I 2020 dataset · failure-type classifier tuned to minimise false alarms (FDR)")

tabs = st.tabs(["📊 Overview", "🔗 Lead indicator", "🎯 Models & threshold",
                "💰 Cost-sensitive", "🚨 Anomaly detection", "🔮 Live prediction"])

df, info = bundle["df"], bundle["info"]
y_test, proba_test = bundle["y_test"], bundle["proba"][model_name]["test"]


def pct(x):
    return f"{x:.1%}"


# ----------------------------------------------------------------------------
# 1. Overview
# ----------------------------------------------------------------------------
with tabs[0]:
    failures = int((df["Failure Type"] != "No Failure").sum())
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Machines (after cleaning)", f"{info['clean_rows']:,}")
    c2.metric("Failures", f"{failures:,}", pct(failures / len(df)))
    c3.metric("Inconsistent rows removed", info["inconsistent_removed"])
    c4.metric("Random failures (RNF) removed", info["rnf_removed"])

    left, right = st.columns(2)
    counts = df["Failure Type"].value_counts().reset_index()
    counts.columns = ["Failure Type", "Count"]
    fig = px.bar(counts, x="Failure Type", y="Count", color="Failure Type", text="Count",
                 title="Class distribution (log scale)", log_y=True)
    fig.update_layout(showlegend=False)
    left.plotly_chart(fig, width="stretch")

    by_type = (df.assign(fail=df["Failure Type"] != "No Failure")
               .groupby("Type")["fail"].mean().reset_index())
    fig = px.bar(by_type, x="Type", y="fail", text=by_type["fail"].map(pct),
                 title="Failure rate by product quality type")
    fig.update_yaxes(tickformat=".0%", title="Failure rate")
    right.plotly_chart(fig, width="stretch")

    st.info(f"A model that always says 'No Failure' scores "
            f"{pct(1 - failures / len(df))} accuracy and catches nothing, so accuracy alone is misleading here.")

    sensor = st.selectbox("Sensor vs failure type", pm.SENSORS + pm.ENGINEERED)
    fig = px.box(df, x="Failure Type", y=sensor, color="Failure Type",
                 category_orders={"Failure Type": ["No Failure"] + list(pm.FAILURE_NAMES.values())})
    fig.update_layout(showlegend=False)
    st.plotly_chart(fig, width="stretch")

# ----------------------------------------------------------------------------
# 2. Lead indicator (bonus)
# ----------------------------------------------------------------------------
with tabs[1]:
    st.subheader("Which sensor is the lead indicator of failure?")
    st.caption("Each failure type is compared with normal machines (correlation with a 0/1 flag). "
               "Raw sensors name the lead; engineered features are shown for comparison.")
    corr = bundle["corr"]
    lead_table, ranking = pm.lead_sensors(corr)

    fig = px.imshow(corr.round(3), text_auto=True, aspect="auto", zmin=-0.3, zmax=0.3,
                    color_continuous_scale="RdBu_r", title="Correlation with each failure type")
    st.plotly_chart(fig, width="stretch")

    a, b = st.columns(2)
    a.markdown("**Lead sensor per failure type**")
    a.dataframe(lead_table.round(3), hide_index=True, width="stretch")
    rank_df = ranking.reset_index()
    rank_df.columns = ["Sensor", "Mean |correlation|"]
    fig = px.bar(rank_df, x="Mean |correlation|", y="Sensor", orientation="h",
                 title="Average strength across failure types")
    fig.update_yaxes(autorange="reversed")
    b.plotly_chart(fig, width="stretch")

    top = ranking.index[0]
    st.success(f"Overall lead indicator: **{top}** (mean |r| = {ranking.iloc[0]:.3f}), "
               f"followed by {ranking.index[1]} ({ranking.iloc[1]:.3f}).")
    st.warning("Correlation is not causation, and every raw sensor is weak on its own: failures come from "
               "combinations (e.g. small temperature gap together with low speed). The dataset has no "
               "time series, so 'lead' means strongest association, not 'changes first'.")

# ----------------------------------------------------------------------------
# 3. Models & threshold
# ----------------------------------------------------------------------------
with tabs[2]:
    st.subheader(f"{model_name} at threshold {threshold:.2f} ({mode})")
    st.caption("Thresholds are chosen on the validation set. The numbers below are on the untouched test set.")

    default_pred = proba_test.argmax(axis=1)
    tuned_pred = pm.predict_with_threshold(proba_test, threshold, nf)
    m_def = pm.failure_metrics(y_test, default_pred, nf)
    m_tun = pm.failure_metrics(y_test, tuned_pred, nf)

    k = st.columns(6)
    k[0].metric("FDR", pct(m_tun["FDR"]), f"{(m_tun['FDR'] - m_def['FDR']) * 100:+.1f} pts vs default",
                delta_color="inverse")
    k[1].metric("Precision", pct(m_tun["Precision"]))
    k[2].metric("Recall", pct(m_tun["Recall"]))
    k[3].metric("False alarms", m_tun["False Positives"], f"{m_tun['False Positives'] - m_def['False Positives']:+d}",
                delta_color="inverse")
    k[4].metric("Missed failures", m_tun["False Negatives"], f"{m_tun['False Negatives'] - m_def['False Negatives']:+d}",
                delta_color="inverse")
    k[5].metric("Accuracy", pct(m_tun["Accuracy"]), f"{(m_tun['Accuracy'] - m_def['Accuracy']) * 100:+.2f} pts")

    compare = pd.DataFrame({"Default (most likely class)": m_def, f"Tuned (threshold {threshold:.2f})": m_tun}).T
    compare = compare[["FDR", "Precision", "Recall", "F1-score", "False Positives",
                       "False Negatives", "Wrong Type", "Accuracy"]]
    st.dataframe(compare.style.format({c: "{:.3f}" for c in ["FDR", "Precision", "Recall", "F1-score", "Accuracy"]}),
                 width="stretch")

    left, right = st.columns(2)
    cm = confusion_matrix(y_test, tuned_pred, labels=range(len(classes)))
    fig = px.imshow(cm, x=classes, y=classes, text_auto=True, color_continuous_scale="Blues",
                    labels=dict(x="Predicted", y="Actual"), title="Confusion matrix (test)")
    left.plotly_chart(fig, width="stretch")

    fig = go.Figure()
    fig.add_scatter(x=val_table["Threshold"], y=val_table["FDR"], name="FDR", mode="lines+markers")
    fig.add_scatter(x=val_table["Threshold"], y=val_table["Recall"], name="Recall", mode="lines+markers")
    fig.add_vline(x=threshold, line_dash="dash", annotation_text="active")
    fig.add_hline(y=min_recall, line_dash="dot", line_color="gray", annotation_text="min recall")
    fig.update_layout(title="Validation: FDR and recall vs threshold", xaxis_title="Threshold", yaxis_tickformat=".0%")
    right.plotly_chart(fig, width="stretch")

    with st.expander("Validation threshold table"):
        st.dataframe(val_table.round(3), hide_index=True, width="stretch")
    with st.expander("Per-class report (test)"):
        rep = classification_report(y_test, tuned_pred, labels=range(len(classes)),
                                    target_names=classes, zero_division=0, output_dict=True)
        st.dataframe(pd.DataFrame(rep).T.round(3), width="stretch")
        tw = classes.index("Tool Wear") if "Tool Wear" in classes else None
        if tw is not None and (tuned_pred == tw).sum() == 0:
            st.warning("No Tool Wear alarms at this threshold: this failure happens at a random time between "
                       "200 and 240 min of wear, and there are very few examples.")

    st.markdown("**Both models at their own FDR-rule threshold (test set)**")
    rows = []
    for name in bundle["models"]:
        vt = pm.threshold_table(bundle["y_val"], bundle["proba"][name]["val"], nf)
        sel = pm.select_fdr_rule(vt, min_recall)
        if sel is None:
            continue
        p = pm.predict_with_threshold(bundle["proba"][name]["test"], sel["Threshold"], nf)
        m = pm.failure_metrics(y_test, p, nf)
        rows.append({"Model": name, "Threshold": sel["Threshold"], "FDR": m["FDR"], "Precision": m["Precision"],
                     "Recall": m["Recall"], "False Positives": m["False Positives"],
                     "False Negatives": m["False Negatives"], "Accuracy": m["Accuracy"]})
    if rows:
        st.dataframe(pd.DataFrame(rows).round(3), hide_index=True, width="stretch")

# ----------------------------------------------------------------------------
# 4. Cost-sensitive
# ----------------------------------------------------------------------------
with tabs[3]:
    st.subheader("Cost-sensitive threshold selection")
    st.caption("Costs come from the sidebar. They are assumptions: replace them with the factory's real numbers.")

    fig = go.Figure()
    for name in bundle["models"]:
        vt = pm.add_cost_column(
            pm.threshold_table(bundle["y_val"], bundle["proba"][name]["val"], nf), c_fa, c_wt, c_mf)
        fig.add_scatter(x=vt["Threshold"], y=vt["Total Cost"], name=name, mode="lines+markers")
    fig.update_layout(title="Total cost vs threshold (validation)", xaxis_title="Threshold",
                      yaxis_title="Total cost")
    st.plotly_chart(fig, width="stretch")

    rows = []
    for name in bundle["models"]:
        vt = pm.threshold_table(bundle["y_val"], bundle["proba"][name]["val"], nf)
        picks = {"FDR rule": pm.select_fdr_rule(vt, min_recall),
                 "Cost rule": pm.select_cost_rule(pm.add_cost_column(vt, c_fa, c_wt, c_mf))}
        for rule, sel in picks.items():
            if sel is None:
                continue
            p = pm.predict_with_threshold(bundle["proba"][name]["test"], sel["Threshold"], nf)
            m = pm.failure_metrics(y_test, p, nf)
            rows.append({"Model": name, "Rule": rule, "Threshold": sel["Threshold"],
                         "False Positives": m["False Positives"], "Wrong Type": m["Wrong Type"],
                         "False Negatives": m["False Negatives"], "Recall": m["Recall"], "FDR": m["FDR"],
                         "Total Cost": pm.total_cost(m, c_fa, c_wt, c_mf)})
    st.markdown("**Test set: FDR rule vs cost rule**")
    res = pd.DataFrame(rows)
    st.dataframe(res.round(3), hide_index=True, width="stretch")

    st.markdown("**How sensitive is the choice to the cost of a missed failure?**")
    ratios = [1, 2, 5, 10, 20, 50]
    rows = []
    for name in bundle["models"]:
        vt = pm.threshold_table(bundle["y_val"], bundle["proba"][name]["val"], nf)
        for r in ratios:
            sel = pm.select_cost_rule(pm.add_cost_column(vt, c_fa, c_wt, c_fa * r))
            p = pm.predict_with_threshold(bundle["proba"][name]["test"], sel["Threshold"], nf)
            m = pm.failure_metrics(y_test, p, nf)
            rows.append({"Model": name, "Missed / false-alarm cost": r, "Threshold": sel["Threshold"],
                         "False Positives": m["False Positives"], "False Negatives": m["False Negatives"],
                         "Test Total Cost": pm.total_cost(m, c_fa, c_wt, c_fa * r)})
    sens = pd.DataFrame(rows)
    fig = px.line(sens, x="Missed / false-alarm cost", y="Test Total Cost", color="Model", markers=True)
    st.plotly_chart(fig, width="stretch")
    with st.expander("Sensitivity table"):
        st.dataframe(sens.round(3), hide_index=True, width="stretch")

# ----------------------------------------------------------------------------
# 5. Anomaly detection
# ----------------------------------------------------------------------------
with tabs[4]:
    st.subheader("Isolation Forest (trained on normal machines only)")
    yb = (y_test != nf).astype(int)
    hist = pd.DataFrame({"Anomaly score": bundle["anomaly"]["test"],
                         "Machine": np.where(yb == 1, "Failure", "Normal")})
    fig = px.histogram(hist, x="Anomaly score", color="Machine", histnorm="probability density",
                       barmode="overlay", nbins=40, opacity=0.6, title="Anomaly score on the test set")
    st.plotly_chart(fig, width="stretch")

    a_table, a_best = pm.anomaly_table(bundle, min_recall)
    a, b = st.columns(2)
    a.markdown("**Validation: alarm threshold as a percentile of normal machines**")
    a.dataframe(a_table.round(3), hide_index=True, width="stretch")
    if a_best is not None:
        res = pm.anomaly_test_result(bundle, a_best["Threshold"])
        b.markdown(f"**Test result at the {int(a_best['Percentile'])}th percentile**")
        b.metric("FDR", pct(res["FDR"]))
        b.metric("Recall", pct(res["Recall"]))
        b.metric("False alarms", res["False Positives"])
        st.error(f"About {res['FDR']:.0%} of its alarms are false: the anomaly detector alone cannot meet the "
                 "FDR requirement. It is useful as a label-free second opinion.")
    else:
        b.warning("No percentile reaches the minimum recall.")

    c, d = st.columns(2)
    c.markdown("**Ranking quality (threshold-free)**")
    c.dataframe(pm.ranking_comparison(bundle).round(3), hide_index=True, width="stretch")
    d.markdown("**Isolation Forest ROC-AUC by failure type**")
    d.dataframe(pm.anomaly_auc_by_type(bundle).round(3), hide_index=True, width="stretch")

# ----------------------------------------------------------------------------
# 6. Live prediction
# ----------------------------------------------------------------------------
PRESETS = {
    "Normal machine": dict(t="M", air=300.0, proc=310.0, rpm=1500, torque=40.0, wear=100),
    "Heat dissipation": dict(t="L", air=302.0, proc=310.5, rpm=1350, torque=45.0, wear=50),
    "Power failure": dict(t="M", air=300.0, proc=310.0, rpm=2600, torque=65.0, wear=80),
    "Overstrain": dict(t="L", air=300.0, proc=310.0, rpm=1300, torque=60.0, wear=190),
    "Tool wear": dict(t="L", air=300.0, proc=310.0, rpm=1500, torque=40.0, wear=225),
}
DEFAULTS = PRESETS["Normal machine"]
for key, value in DEFAULTS.items():
    st.session_state.setdefault(f"in_{key}", value)


def apply_preset(name):
    for key, value in PRESETS[name].items():
        st.session_state[f"in_{key}"] = value


with tabs[5]:
    st.subheader("Enter machine readings and get a prediction")
    st.caption(f"Using {model_name} with threshold {threshold:.2f} ({mode}) from the sidebar.")

    st.markdown("**Quick examples**")
    cols = st.columns(len(PRESETS))
    for col, name in zip(cols, PRESETS):
        col.button(name, on_click=apply_preset, args=(name,), width="stretch")

    c1, c2, c3 = st.columns(3)
    t_in = c1.selectbox("Product quality type", ["L", "M", "H"], key="in_t")
    air = c1.number_input("Air temperature [K]", 290.0, 310.0, step=0.1, key="in_air")
    proc = c2.number_input("Process temperature [K]", 300.0, 320.0, step=0.1, key="in_proc")
    rpm = c2.number_input("Rotational speed [rpm]", 1000, 3000, step=10, key="in_rpm")
    torque = c3.number_input("Torque [Nm]", 0.0, 100.0, step=0.5, key="in_torque")
    wear = c3.number_input("Tool wear [min]", 0, 300, step=1, key="in_wear")

    row = {"Type": t_in, "Air temperature [K]": air, "Process temperature [K]": proc,
           "Rotational speed [rpm]": rpm, "Torque [Nm]": torque, "Tool wear [min]": wear}
    X_in = pm.build_input_frame([row])
    proba, pred, labels, anomaly_score = pm.predict_frame(bundle, model_name, X_in, threshold)
    label, p_fail = labels[0], 1 - proba[0][nf]

    st.divider()
    if pred[0] == nf:
        st.success("### ✅ No alarm: machine looks healthy")
    else:
        st.error(f"### 🚨 Alarm: predicted **{label}**")

    r1, r2 = st.columns([1, 1])
    prob_df = pd.DataFrame({"Class": classes, "Probability": proba[0]})
    fig = px.bar(prob_df, x="Probability", y="Class", orientation="h", text=prob_df["Probability"].map(pct),
                 title="Class probabilities")
    fig.add_vline(x=threshold, line_dash="dash", annotation_text="alarm threshold")
    fig.update_xaxes(range=[0, 1], tickformat=".0%")
    r1.plotly_chart(fig, width="stretch")

    r2.metric("Total failure probability", pct(p_fail))
    a_thr = a_best["Threshold"] if a_best is not None else np.percentile(bundle["anomaly"]["val"], 95)
    r2.metric("Anomaly score (Isolation Forest)", f"{anomaly_score[0]:.3f}",
              "unusual operating point" if anomaly_score[0] >= a_thr else "typical operating point",
              delta_color="inverse" if anomaly_score[0] >= a_thr else "normal")
    feat = X_in.iloc[0]
    r2.markdown(f"Temp difference **{feat['Temp difference [K]']:.1f} K** · Power **{feat['Power [W]']:,.0f} W** · "
                f"Wear × Torque **{feat['Wear x Torque']:,.0f}**")

    st.markdown("**Physical rule check** (documented AI4I failure rules, independent of the model)")
    checks = pm.physical_checks(row)
    st.dataframe(checks.style.map(lambda v: "background-color: #f8d7da" if v is True else "", subset=["Triggered"]),
                 hide_index=True, width="stretch")
    if bool(checks["Triggered"].any()) != (pred[0] != nf):
        st.info("The model and the physical rules disagree. The model only alarms when its confidence passes "
                "the threshold, so borderline readings can differ.")

    with st.expander("📁 Batch prediction from a CSV"):
        need = ["Type"] + pm.SENSORS
        template = pd.DataFrame([row])[need]
        st.download_button("Download template", template.to_csv(index=False), "template.csv", "text/csv")
        batch = st.file_uploader("CSV with columns: " + ", ".join(need), type="csv", key="batch")
        if batch is not None:
            data = pd.read_csv(batch)
            missing = [c for c in need if c not in data.columns]
            if missing:
                st.error(f"Missing columns: {missing}")
            else:
                Xb = pm.build_input_frame(data[need].to_dict("records"))
                pb, pr, lb, an = pm.predict_frame(bundle, model_name, Xb, threshold)
                out = data.copy()
                out["Prediction"] = lb
                out["Failure probability"] = (1 - pb[:, nf]).round(3)
                out["Anomaly score"] = an.round(3)
                st.dataframe(out, hide_index=True, width="stretch")
                st.caption(f"{int((pr != nf).sum())} alarm(s) out of {len(out)} machines.")
                st.download_button("Download predictions", out.to_csv(index=False), "predictions.csv", "text/csv")


Overwriting app.py


## Sanity check
Before launching the dashboard, train the models once and print the key results. They should match the main notebook (same seeds, same split).

In [10]:
import pandas as pd
import pm_core as pm

data_path = pm.find_dataset()
print("Dataset:", data_path)

bundle = pm.train_bundle(pd.read_csv(data_path))
nf, classes = bundle["nf"], bundle["class_names"]

print("\nCleaning:", bundle["info"])
print("Classes:", classes)
print(bundle["df"]["Failure Type"].value_counts().to_string())

# FDR rule: lowest FDR on validation with recall >= 70%, then evaluate once on test
rows = []
for name in bundle["models"]:
    val_table = pm.threshold_table(bundle["y_val"], bundle["proba"][name]["val"], nf)
    best = pm.select_fdr_rule(val_table, 0.70)
    pred = pm.predict_with_threshold(bundle["proba"][name]["test"], best["Threshold"], nf)
    rows.append({"Model": name, "Threshold": best["Threshold"], **pm.failure_metrics(bundle["y_test"], pred, nf)})
display(pd.DataFrame(rows).round(3))

lead_table, ranking = pm.lead_sensors(bundle["corr"])
display(lead_table.round(3))
print("Lead indicator overall:", ranking.index[0])

Dataset: C:\Users\fayro\OneDrive\Desktop\industrial-predictive-maintenance\data\ai4i2020.csv

Cleaning: {'raw_rows': 10000, 'clean_rows': 9972, 'inconsistent_removed': 9, 'rnf_removed': 19, 'multi_mode_rows': 23}
Classes: ['Heat Dissipation', 'No Failure', 'Overstrain', 'Power Failure', 'Tool Wear']
Failure Type
No Failure          9643
Heat Dissipation     115
Power Failure         91
Overstrain            78
Tool Wear             45


,Model,Threshold,Precision,Recall,F1-score,False Positives,Wrong Type,False Negatives,FDR,Accuracy,Alarms
0,Random Forest,0.7,0.904,0.712,0.797,5,0,19,0.096,0.988,52
1,XGBoost,0.9,0.844,0.818,0.831,10,0,12,0.156,0.989,64


,Failure Type,Lead sensor,Correlation
0,Tool Wear,Tool wear [min],0.118
1,Heat Dissipation,Torque [Nm],0.153
2,Power Failure,Rotational speed [rpm],0.131
3,Overstrain,Torque [Nm],0.161


Lead indicator overall: Torque [Nm]


## Launch the dashboard

Run the next cell, then open **http://localhost:8501** in your browser (it also tries to show it below). Training takes about 10 seconds the first time; after that the models are cached.

Prefer the terminal? Run `streamlit run app.py` from this folder instead.

*Google Colab:* the dashboard cannot be opened at `localhost` there. Run `!npx localtunnel --port 8501` in a second cell and use the printed link (the password is the output of `!curl ipv4.icanhazip.com`).

In [11]:
import subprocess, sys, time
from IPython.display import IFrame, display

server = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501", "--server.headless", "true"]
)
time.sleep(6)
print("Dashboard running at http://localhost:8501")
display(IFrame("http://localhost:8501", width="100%", height=900))

Dashboard running at http://localhost:8501


Run this cell when you are done to stop the server:

In [12]:
server.terminate()
print("Dashboard stopped.")

Dashboard stopped.
